# Block 1 — Train a ship detector (Colab, A100)

Pipeline of the whole repo: **satellite image → onboard ship detection (tiled) → compact binary alert → ground decoder.**
This notebook only does the first part: turn the Airbus Ship Detection masks into boxes and fine-tune YOLOv8s.

Outputs (all written to Google Drive, `MyDrive/ship_demo/`, so they survive a runtime disconnect):
- `v8s_768/weights/best.pt` — the checkpoint used locally for inference
- `v8s_768/results.csv` + plots — training curves
- `metrics.json` — final val **and** test metrics
- `label_check.png`, `test_predictions/` — visual checks
- `val_test_subset.zip` — val + test images and labels, needed locally for Blocks 2 and 4

## Cell 0 — Check the hardware
**Why:** the training settings below (`batch=32`, `cache='ram'`) assume an A100 and enough RAM.
`cache='ram'` keeps ~8.8k decoded 768×768 images in memory (~16 GB) so the GPU isn't waiting on JPEG decoding.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import psutil, shutil
vm = psutil.virtual_memory()
print(f"RAM: {vm.total/1e9:.1f} GB total, {vm.available/1e9:.1f} GB free")
print(f"Disk /content: {shutil.disk_usage('/content').free/1e9:.1f} GB free")

## Cell 1 — Install Ultralytics, mount Drive
**Why:** Ultralytics provides YOLOv8 training/validation/export. We print the exact version so the local
laptop installs the **same** one (a checkpoint from one version can misbehave in another).
Drive is mounted only for *outputs*; the data stays on the fast local disk.

In [ ]:
!pip install -q ultralytics
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 — Paths and constants, check the data
**Why:** the data was downloaded with `kagglehub` to the runtime's local disk. If the runtime was reset,
that folder is gone, so we re-download it automatically (~6 min + extraction).

- `N_SHIP=8000, N_EMPTY=800`: ships + ~10% empty images (recipe from the HF ship-detection 1st place).
- `MIN_SIDE=3`: boxes thinner than 3 px are dropped as degenerate (mostly annotation noise).
- `SEED=0`: every random choice is reproducible.

In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("/root/.cache/kagglehub/competitions/airbus-ship-detection")
if not (DATA / "train_ship_segmentations_v2.csv").exists():
    import kagglehub
    DATA = Path(kagglehub.competition_download("airbus-ship-detection"))

IMG_DIR = DATA / "train_v2"
CSV     = DATA / "train_ship_segmentations_v2.csv"
OUT     = Path("/content/drive/MyDrive/ship_demo")  # Drive: survives disconnects
WORK    = Path("/content/ships")                    # local disk: fast reads during training
OUT.mkdir(parents=True, exist_ok=True)

N_SHIP, N_EMPTY = 8000, 800
MIN_SIDE = 3
SEED = 0
IMG_SIZE = 768

print(sorted(p.name for p in DATA.iterdir()))
print("train_v2 images:", sum(1 for _ in IMG_DIR.iterdir()))
df = pd.read_csv(CSV)
print("CSV rows:", len(df))
df.head()

## Cell 3 — Pick the subset and split 80/10/10
**Why:**
- The CSV has **one row per ship**; an image with no ship has one row with an empty `EncodedPixels`.
  So "image has a ship" = "at least one of its rows is non-empty".
- Most images are empty; training on all of them would waste time teaching "water".
  But we keep ~10% empty images so the model learns what is *not* a ship (waves, clouds, coast).
- **Train** = learn, **val** = pick `best.pt` during training (+ INT8 calibration later),
  **test** = untouched until the end → unbiased final numbers.
- The split is done separately for ship and empty images (stratified), so each set keeps ~10% empty.
- Caveat for the README: some Airbus images are overlapping crops of the same scene, so a random
  split can leak near-duplicates between sets → scores may be optimistic.

In [ ]:
has_ship = df["EncodedPixels"].notna().groupby(df["ImageId"]).any()
ship_ids  = has_ship[has_ship].index.to_series()
empty_ids = has_ship[~has_ship].index.to_series()
print(f"images with ships: {len(ship_ids)} | empty: {len(empty_ids)} | ship rows: {df['EncodedPixels'].notna().sum()}")

def three_way(ids):
    # ids are already in random order (from .sample), so slicing = random split
    ids = ids.tolist()
    a, b = int(0.8 * len(ids)), int(0.9 * len(ids))
    return ids[:a], ids[a:b], ids[b:]

ship_sel  = ship_ids.sample(N_SHIP, random_state=SEED)
empty_sel = empty_ids.sample(N_EMPTY, random_state=SEED)

splits = {}
for name, s_part, e_part in zip(["train", "val", "test"], three_way(ship_sel), three_way(empty_sel)):
    splits[name] = s_part + e_part
for name, ids in splits.items():
    print(name, len(ids))

## Cell 4 — Decode RLE masks into boxes
**Why / how:**
- Each mask is run-length encoded: pairs `(start, length)`, starts **1-indexed**, pixels numbered
  **top-to-bottom, then left-to-right** (column-major). So we fill a flat array and reshape it with
  `order="F"` (Fortran = column order). Getting this wrong mirrors every box across the diagonal —
  Cell 5 checks it visually.
- One CSV row = one ship, so one mask → one axis-aligned box (min/max of its rows and columns).
- Boxes with width or height `< MIN_SIDE` are dropped; we print how many so the README has the real count.
- Unreadable or wrong-size images are skipped instead of crashing.

In [ ]:
import cv2
import numpy as np
from tqdm.auto import tqdm

def rle_decode(rle, shape=(IMG_SIZE, IMG_SIZE)):
    s = np.asarray(rle.split(), dtype=np.int64)
    starts, lengths = s[0::2] - 1, s[1::2]      # -1: CSV starts are 1-indexed
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for st, ln in zip(starts, lengths):
        flat[st:st + ln] = 1
    return flat.reshape(shape, order="F")       # column-major numbering

def mask_to_box(mask):
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    # +1 makes x1/y1 exclusive, so width = x1 - x0 counts pixels correctly
    return int(cols[0]), int(rows[0]), int(cols[-1]) + 1, int(rows[-1]) + 1

rles = df[df["EncodedPixels"].notna()].groupby("ImageId")["EncodedPixels"].apply(list)

boxes, dropped, bad = {}, [], []
for name in splits:
    kept = []
    for img_id in tqdm(splits[name], desc=name):
        img = cv2.imread(str(IMG_DIR / img_id))
        if img is None or img.shape[:2] != (IMG_SIZE, IMG_SIZE):
            bad.append(img_id)
            continue
        img_boxes = []
        for rle in rles.get(img_id, []):
            x0, y0, x1, y1 = mask_to_box(rle_decode(rle))
            if (x1 - x0) < MIN_SIDE or (y1 - y0) < MIN_SIDE:
                dropped.append((img_id, x1 - x0, y1 - y0))
                continue
            img_boxes.append((x0, y0, x1, y1))
        boxes[img_id] = img_boxes
        kept.append(img_id)
    splits[name] = kept

all_boxes = [b for bs in boxes.values() for b in bs]
print(f"unreadable/wrong-size images skipped: {len(bad)} {bad[:5]}")
print(f"boxes kept: {len(all_boxes)} | dropped (side < {MIN_SIDE}px): {len(dropped)} "
      f"({100 * len(dropped) / max(1, len(dropped) + len(all_boxes)):.2f}%)")
long_side = np.array([max(x1 - x0, y1 - y0) for x0, y0, x1, y1 in all_boxes])
print("ship long side in px, percentiles 5/25/50/75/95:", np.percentile(long_side, [5, 25, 50, 75, 95]).round(1))

pd.DataFrame([(i, n) for n, ids in splits.items() for i in ids],
             columns=["ImageId", "split"]).to_csv(OUT / "split.csv", index=False)

## Cell 5 — Visual sanity check  ⚠️ STOP and look before running further
**What to look for:** red = decoded mask, green = box. The green box must tightly wrap the red ship.
If the boxes sit on empty water or look mirrored across the diagonal, the RLE order is wrong.

In [ ]:
import matplotlib.pyplot as plt

examples = [i for i in splits["train"] if len(boxes[i]) >= 2][:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_id in zip(axes.flat, examples):
    img = cv2.cvtColor(cv2.imread(str(IMG_DIR / img_id)), cv2.COLOR_BGR2RGB)
    mask = np.zeros(img.shape[:2], dtype=bool)
    for rle in rles[img_id]:
        mask |= rle_decode(rle).astype(bool)
    overlay = img.copy()
    overlay[mask] = (255, 0, 0)
    img = cv2.addWeighted(img, 0.6, overlay, 0.4, 0)
    for x0, y0, x1, y1 in boxes[img_id]:
        cv2.rectangle(img, (x0, y0), (x1, y1), (0, 255, 0), 2)
    ax.imshow(img); ax.set_title(f"{img_id} ({len(boxes[img_id])} ships)"); ax.axis("off")
plt.tight_layout()
plt.savefig(OUT / "label_check.png", dpi=80)
plt.show()

## Cell 6 — Build the YOLO dataset folder
**Why:** Ultralytics expects `images/<split>/x.jpg` with a matching `labels/<split>/x.txt`.
Each label line is `class cx cy w h`, all **normalised to 0–1** by the image size (so labels don't depend on resolution).
Empty images get an empty `.txt` → Ultralytics treats them as background.

We **copy** the ~8.8k images (~1–2 GB, local disk to local disk) instead of symlinking: Ultralytics finds
labels by replacing `images` with `labels` in the path, and copying removes any doubt about how it resolves links.

In [ ]:
import shutil

if WORK.exists():
    shutil.rmtree(WORK)
for name, ids in splits.items():
    (WORK / "images" / name).mkdir(parents=True)
    (WORK / "labels" / name).mkdir(parents=True)
    for img_id in tqdm(ids, desc=name):
        shutil.copy(IMG_DIR / img_id, WORK / "images" / name / img_id)
        lines = [f"0 {(x0 + x1) / 2 / IMG_SIZE:.6f} {(y0 + y1) / 2 / IMG_SIZE:.6f} "
                 f"{(x1 - x0) / IMG_SIZE:.6f} {(y1 - y0) / IMG_SIZE:.6f}"
                 for x0, y0, x1, y1 in boxes[img_id]]
        (WORK / "labels" / name / f"{Path(img_id).stem}.txt").write_text("\n".join(lines))

(WORK / "data.yaml").write_text(
    f"path: {WORK}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: ship\n")
print((WORK / "data.yaml").read_text())
for name, ids in splits.items():
    n_pos = sum(1 for i in ids if boxes[i])
    print(f"{name}: {len(ids)} images, {n_pos} with ships, {len(ids) - n_pos} empty")

## Cell 7 — Train YOLOv8s (~45 min on the A100)
**Settings and why:**
- `yolov8s.pt`: COCO-pretrained small model (11.2M params). Fine-tuning reuses its edge/shape features.
- `imgsz=768`: native Airbus size, so no downscaling (small ships would shrink to nothing).
- `time=0.75`: hard 45-min budget. It **overrides `epochs`**: Ultralytics re-plans the number of epochs
  and the learning-rate schedule to fit the time.
- `batch=32`: fits comfortably in A100 memory at 768.
- `cache="ram"`: decode images once, keep them in RAM (Ultralytics checks free RAM and warns if short).
- `flipud=0.5`: overhead imagery has no "up", so vertical flips are valid. On top of the default
  augmentations (mosaic, colour jitter, scale/shift, left-right flip).
- `seed=0`: reproducibility. `project=OUT`: weights and curves go to Drive every epoch.

**If the runtime disconnects:** re-run Cells 0–6, then
`YOLO(OUT / "v8s_768/weights/last.pt").train(resume=True)`.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.train(
    data=str(WORK / "data.yaml"),
    imgsz=IMG_SIZE,
    epochs=100,      # upper bound only; `time` decides the real number
    time=0.75,       # hours
    batch=32,
    cache="ram",
    flipud=0.5,
    seed=SEED,
    workers=8,
    project=str(OUT),
    name="v8s_768",
    exist_ok=True,
)

## Cell 8 — Final metrics, example predictions, export for the laptop
**Why:**
- `val` numbers are slightly optimistic (val chose `best.pt`); **`test` is the number for the README**.
- mAP50 = average precision with a match threshold of IoU ≥ 0.5; mAP50-95 averages over IoU 0.5…0.95
  (stricter: rewards tight boxes). P/R = precision/recall at the best-F1 confidence.
- The zip holds val + test images and labels for local work: test → synthetic frame (Block 2) and
  accuracy comparison (Block 4); val → INT8 calibration. Stored uncompressed because JPEGs are already compressed.

**Download to the laptop:** `v8s_768/weights/best.pt`, `val_test_subset.zip`, `metrics.json`,
`v8s_768/results.png`, `label_check.png`, `test_predictions/`.

In [ ]:
import json, zipfile

best = YOLO(OUT / "v8s_768" / "weights" / "best.pt")
metrics = {}
for name in ["val", "test"]:
    m = best.val(data=str(WORK / "data.yaml"), split=name, imgsz=IMG_SIZE, batch=32,
                 project=str(OUT), name=f"eval_{name}", exist_ok=True)
    metrics[name] = {"mAP50": round(float(m.box.map50), 4), "mAP50-95": round(float(m.box.map), 4),
                     "precision": round(float(m.box.mp), 4), "recall": round(float(m.box.mr), 4),
                     "images": len(splits[name])}
    print(name, metrics[name])
(OUT / "metrics.json").write_text(json.dumps(metrics, indent=2))

# First 8 test images that contain ships
sample = [str(WORK / "images" / "test" / i) for i in splits["test"] if boxes[i]][:8]
best.predict(sample, imgsz=IMG_SIZE, conf=0.25, save=True,
             project=str(OUT), name="test_predictions", exist_ok=True)

zpath = OUT / "val_test_subset.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
    for name in ["val", "test"]:
        for img_id in splits[name]:
            stem = Path(img_id).stem
            z.write(WORK / "images" / name / img_id, f"{name}/images/{img_id}")
            z.write(WORK / "labels" / name / f"{stem}.txt", f"{name}/labels/{stem}.txt")
    z.write(OUT / "split.csv", "split.csv")
print(f"{zpath.name}: {zpath.stat().st_size / 1e6:.0f} MB")